# Session 19: Feature Engineering I

This notebook performs first-stage feature engineering by one-hot
encoding categorical variables.

**Automatically selected source dataset:** `data/student-mat.csv`

**Original shape:** `(395, 33)`

**Encoded shape:** `(395, 42)`

**Categorical columns encoded:** `17`

**Dummy columns created:** `26`


In [ ]:
from pathlib import Path

import pandas as pd


In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_FILE = PROJECT_ROOT / "data/student-mat.csv"
OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "student_data_encoded.csv"
)

df = pd.read_csv(
    INPUT_FILE,
    sep=None,
    engine="python",
)

print("Input file:", INPUT_FILE)
print("Original shape:", df.shape)
df.head()


In [ ]:
cat_cols = df.select_dtypes(
    include=["object", "category"]
).columns

print("Categorical columns:")
print(list(cat_cols))
print("Number of categorical columns:", len(cat_cols))

df_encoded = pd.get_dummies(
    df,
    columns=list(cat_cols),
    drop_first=True,
)

print("Encoded shape:", df_encoded.shape)


In [ ]:
boolean_columns = (
    df_encoded
    .select_dtypes(include="bool")
    .columns
)

if len(boolean_columns) > 0:
    df_encoded[boolean_columns] = (
        df_encoded[boolean_columns]
        .astype("int8")
    )

remaining_object_columns = (
    df_encoded
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

assert df_encoded.shape[0] == df.shape[0]
assert df_encoded.index.equals(df.index)
assert len(remaining_object_columns) == 0
assert not df_encoded.columns.duplicated().any()

if "G3" in df.columns:
    assert "G3" in df_encoded.columns
    assert df_encoded["G3"].equals(df["G3"])

print("All one-hot encoding sanity checks passed.")


In [ ]:
OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

df_encoded.to_csv(
    OUTPUT_FILE,
    index=False,
)

df_encoded_check = pd.read_csv(OUTPUT_FILE)

assert df_encoded_check.shape == df_encoded.shape
assert (
    df_encoded_check.columns.tolist()
    == df_encoded.columns.tolist()
)

print("Encoded dataset saved to:")
print(OUTPUT_FILE)
print("Saved shape:", df_encoded_check.shape)

df_encoded_check.head()


## Prompt-Engineered Explanation of One-Hot Encoding

### Summary

The categorical variables were replaced by binary dummy variables. The
number of rows remained `395`, while the number of columns
changed from `33` to `42`.

### Interpretation

Each categorical variable with \(k\) categories produces \(k-1\) dummy
columns because `drop_first=True` was used. The omitted category becomes
the reference category. A row containing zero across all dummy columns
associated with an original variable represents that reference category.

Using `drop_first=True` removes redundant information and helps prevent
perfect multicollinearity, commonly called the dummy-variable trap, in
regression models that include an intercept.

### Recommendation

Verify that the row count is unchanged, no object columns remain, dummy
columns contain only zero and one, duplicate columns are absent, and the
`G3` target remains unchanged.

Verification note: The explanation was checked against the actual
DataFrame shapes, categorical-column list, dummy-column names, data
types, row count, and target-variable values.
